In [7]:
%load_ext jupyter_black

The jupyter_black extension is already loaded. To reload it, use:
  %reload_ext jupyter_black


In [8]:
import os
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [9]:
in_dir = "../../results/02_10x_visium/01_filter_data"
anno_dir = "../../results/09_analysis_podocyte_niche/02_identify_podocyte_niche/update"
out_dir = "../../results/09_analysis_podocyte_niche/08_identify_Periglomerular"
os.makedirs(out_dir, exist_ok=True)

In [10]:
sample_list = [
    "15837",
    "15852",
    "16512",
    "21022",
    "8262",
    "B23-03794-5",
    "B23-05785-7-1",
    "B23-05785-7-2",
    "X1842",
    "X1845",
    "X1899",
    "X1912",
]

In [11]:
df_count_list, df_obs = [], []
for sample in sample_list:
    print(f"Processing sample {sample}...")
    adata = sc.read_h5ad(f"{in_dir}/{sample}.h5ad")

    # add annotation
    df_anno = pd.read_csv(f"{anno_dir}/{sample}_1.csv", index_col=0)
    adata.obs = adata.obs.merge(df_anno, left_index=True, right_index=True)

    adata = adata[adata.obs["in_tissue"] == 1].copy()
    sq.gr.spatial_neighbors(adata)
    adata.obs["annotation_v2"] = adata.obs["annotation"].copy()

    for spot in adata.obs_names[adata.obs["annotation"] == "Glomerular"]:
        idx = adata.obs_names.get_loc(spot)
        neighbors = adata.obsp["spatial_connectivities"][idx].nonzero()[1]

        # set annotation of neighbors to "Periglomerular"
        adata.obs.loc[adata.obs_names[neighbors], "annotation_v2"] = "Periglomerular"

        # update unassigned to "Tubulointerstitial"
        adata.obs["annotation_v2"] = adata.obs["annotation_v2"].str.replace(
            "unassigned", "Tubulointerstitial"
        )

    # re-assign annotation_v2 for spots with annotation "Glomerular"
    adata.obs.loc[adata.obs["annotation"] == "Glomerular", "annotation_v2"] = (
        "Glomerular"
    )

    # visualize
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    sq.pl.spatial_scatter(adata, color="annotation_v2", ax=ax)
    plt.savefig(f"{out_dir}/{sample}_spatial_annotation_v2.png", dpi=300)
    plt.close()

    adata.write_h5ad(f"{out_dir}/{sample}.h5ad")

Processing sample 15837...
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
Processing sample 15852...
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
Processing sample 16512...
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
Processing sample 21022...
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
Processing sample 8262...
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
Processing sample B23-03794-5...
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
Processing sample B23-05785-7-1...
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.              